In [1]:
!pip install uv
!uv pip install lambeq pandas tqdm pennylane torch clip mlflow cotengra optuna
!pip install git+https://github.com/openai/CLIP.git

Using Python 3.12.12 environment at: /usr
Audited 9 packages in 100ms
  Cloning https://github.com/openai/CLIP.git to /tmp/pip-req-build-_8yxv510
  Running command git clone --filter=blob:none --quiet https://github.com/openai/CLIP.git /tmp/pip-req-build-_8yxv510
  Resolved https://github.com/openai/CLIP.git to commit ded190a052fdf4585bd685cee5bc96e0310d2c93
  Preparing metadata (setup.py) ... done


In [1]:
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
from tqdm import trange, tqdm
import pandas as pd
import numpy as np
import torch, os, clip, pickle, mlflow, time, sys

root_path = os.getcwd()

/Users/tls/qml/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
os.chdir(root_path)

from modules.data_processing import *
from modules.model import *

txt_path = os.path.join(root_path, 'text/aro/attribution')
img_path = os.path.join(root_path, 'images/aro_512/attribution')

In [ ]:
OBMAP = (1,1,1)
NLAYERS = 2
train_einsum = load_pkl(os.path.join(txt_path, 'aro_train_einsum_as_12.pkl'))
valid_einsum = load_pkl(os.path.join(txt_path, 'aro_valid_einsum_as_12.pkl'))
test_einsum = load_pkl(os.path.join(txt_path, 'aro_test_einsum_as_12.pkl'))

IMG_DIM = 512
# train_img = load_pkl(os.path.join(img_path, 'att_imgenc_train_512.pkl'))
# valid_img = load_pkl(os.path.join(img_path, 'att_imgenc_valid_512.pkl'))
# test_img = load_pkl(os.path.join(img_path, 'att_imgenc_test_512.pkl'))
train_img = load_pkl(os.path.join(img_path, 'aro_train_tns.pkl'))
valid_img = load_pkl(os.path.join(img_path, 'aro_valid_tns.pkl'))
test_img = load_pkl(os.path.join(img_path, 'aro_test_tns.pkl'))

In [4]:
train_pos_einsum, train_neg_einsum = zip(*train_einsum)
valid_pos_einsum, valid_neg_einsum = zip(*valid_einsum)
test_pos_einsum, test_neg_einsum = zip(*test_einsum)
train_img = tuple(train_img)
valid_img = tuple(valid_img)
test_img = tuple(test_img)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_default_device(device)
torch.set_default_dtype(torch.float64)

model = EinsumModel()
model.precision = torch.complex128
# model.from_einsum(train_pos_einsum + train_neg_einsum + valid_pos_einsum + valid_neg_einsum + test_pos_einsum + test_neg_einsum)
model.from_einsum(train_pos_einsum + train_neg_einsum + valid_pos_einsum + valid_neg_einsum + test_pos_einsum + test_neg_einsum + train_img + valid_img + test_img)

model.initialise_weights(near_id=False)
model.to(device)
TPARAMS = sum(p.numel() for p in model.parameters() if p.requires_grad)

if next(model.parameters()).is_cuda and torch.cuda.is_available():
    print("Model is on GPU")
else:
    print("Running on CPU...")
print(f"Parameters: #{TPARAMS}")

100%|██████████| 86211/86211 [00:00<00:00, 96429.95it/s] 


Running on CPU...
Parameters: #54100


In [ ]:
BATCH_SIZE = 256
N_EXAMPLES = len(train_einsum)

generator = torch.Generator(device=device)
train_dataset = CLIP_Dataset(train_pos_einsum, train_neg_einsum, train_img)
train_dataset = subsample_data(train_dataset)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn, generator=generator)

valid_dataset = CLIP_Dataset(valid_pos_einsum, valid_neg_einsum, valid_img)
valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE, collate_fn=collate_fn, generator=generator)

test_dataset = CLIP_Dataset(test_pos_einsum, test_neg_einsum, test_img)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, collate_fn=collate_fn, generator=generator)

# model.compile_dataset(train_loader, 'aro')
# model.compile_dataset(valid_loader, 'aro')
# model.compile_dataset(test_loader, 'aro')
model.compile_dataset2(train_loader, 'aro')
model.compile_dataset2(valid_loader, 'aro')
model.compile_dataset2(test_loader, 'aro')
print(f"Unique Contractions: {len(model.path_cache)}")

Unique Contractions: 149


In [8]:
dtime = time.strftime("%m_%d_%H:%M:%S")
fpath = os.path.join(root_path, 'aro_runs', dtime)
if not os.path.exists(fpath):
    os.makedirs(fpath)
db_path = os.path.join(root_path, 'mlf.db')
mlflow.pytorch.autolog()
mlflow.set_tracking_uri(f"sqlite:///{db_path}")
mlflow.set_experiment('qclip_aro_att')

<Experiment: artifact_location='/Users/tls/Desktop/Work/multi_modal/qclip/mlruns/1', creation_time=1781798752011, experiment_id='1', last_update_time=1781798752011, lifecycle_stage='active', name='qclip_aro_att', tags={}, trace_location=None, workspace='default'>

In [9]:
EPOCHS = 100
LEARNING_RATE = 1e-2
LR_FACTOR = 0.5
MPATIENCE = 3
SEED = int.from_bytes(os.urandom(4))
set_seed(SEED)

optimizer = torch.optim.AdamW([{'params': model.weights, 'lr': LEARNING_RATE}], weight_decay=1e-5)
# {'params': model.word_importance, 'lr': LEARNING_RATE/10}, {'params': model.log_temp, 'lr': 0.01}], weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=LR_FACTOR, patience=5)
loss_fn = QInfoNCE_cos(device=device)
acc_fn = fs_distance

In [ ]:
trajectory = []
initial_params = torch.cat([p.view(-1) for p in model.parameters()]).detach().clone()
print(f"Starting training at: {dtime}")

with mlflow.start_run(run_name=dtime):
    # --- Log Hyperparameters ---
    mlflow.log_param("seed", SEED)
    mlflow.log_param("learning_rate_factor", LR_FACTOR)
    mlflow.log_param("patience", MPATIENCE)
    mlflow.log_param("learning_rate", LEARNING_RATE)
    mlflow.log_param("batch_size", BATCH_SIZE)
    mlflow.log_param("total_parameters", TPARAMS)
    mlflow.log_param("type_qubits", OBMAP)
    mlflow.log_param("ansatze_layers", NLAYERS)


    # --- Start Training ---
    for epoch in trange(EPOCHS):
        # --- Training Pass ---
        train_start = time.time()
        # train_loss, train_acc, avg_grad_norm = update_model_aro(model, train_loader, loss_fn, acc_fn, optimizer)
        train_loss, train_acc, avg_grad_norm = update_model_aro2(model, train_loader, loss_fn, acc_fn, optimizer)
        train_end = time.time()

        # --- Validation Pass ---
        valid_start = time.time()
        # valid_acc = eval_model_aro(model, valid_loader, acc_fn)
        valid_acc = eval_model_aro2(model, valid_loader, acc_fn)
        valid_end = time.time()

        # --- Record Time & Save Current Weights ---
        model.save(os.path.join(fpath, 'model.lt'))
        train_time = train_end - train_start
        valid_time = valid_end - valid_start

        # --- Compute & Log Relevant Metrics ---
        current_lr = optimizer.param_groups[0]["lr"]
        current_w = torch.nn.utils.parameters_to_vector(model.parameters()).detach().cpu()
        trajectory.append(current_w)
        init_mad = torch.mean(torch.abs(current_w - initial_params)).item()
        theta_var = torch.mean(torch.abs(current_w - torch.mean(current_w))).item()

        mlflow.log_metric("train_loss", train_loss, step=epoch+1)
        mlflow.log_metric("train_accuracy", train_acc, step=epoch+1)
        mlflow.log_metric("valid_accuracy", valid_acc, step=epoch+1)
        mlflow.log_metric("avg_grad_norm", avg_grad_norm, step=epoch+1)
        mlflow.log_metric("epoch_time", train_time + valid_time, step=epoch+1)
        mlflow.log_metric("learning_rate", current_lr, step=epoch+1)
        mlflow.log_metric("temperature", model.temp, step=epoch+1)
        mlflow.log_metric("parameter_drift", init_mad, step=epoch+1)
        mlflow.log_metric("parameter_variance", theta_var, step=epoch+1)

        # --- Print Current Epoch Information ---
        tqdm.write(f"Epoch {epoch+1}: Loss {train_loss:.4f}, Train Acc {train_acc:.4f}, Val Acc {valid_acc:.4f}, Avg Grad Norm {avg_grad_norm:.2e}, Train Time {train_time/60:.2f}, Valid Time {valid_time/60:.2f} (LR: {current_lr}, T: {model.temp.item():.5f}), Drift: {init_mad:.4f}, Variance: {theta_var:.4f}".strip(), file=sys.stderr)

        if (current_lr <= LEARNING_RATE*LR_FACTOR**MPATIENCE) or (epoch + 1 == EPOCHS):
            mlflow.log_param("epochs", epoch + 1)
            break
        else:
            scheduler.step(valid_acc)

    # --- Evaluate Model on Test Set ---
    # test_acc = eval_model_aro(model, test_loader, acc_fn)
    test_acc = eval_model_aro2(model, test_loader, acc_fn)
    mlflow.log_metric("test_accuracy", test_acc, step=epoch+1)
    print(f"Test Accuracy: {test_acc}")

store_pkl(trajectory, os.path.join(fpath,'trj.pkl'))
mlflow.end_run()

Starting training at: 06_18_17:10:50


Epoch 1: Loss 5.5693, Train Acc 0.5527, Val Acc 0.5070, Avg Grad Norm 1.54e-01, Train Time 0.77, Valid Time 0.05 (LR: 0.01, T: 0.07000), Drift: 0.0549, Variance: 0.7856
Epoch 2: Loss 5.4534, Train Acc 0.6737, Val Acc 0.5198, Avg Grad Norm 1.74e-01, Train Time 0.71, Valid Time 0.05 (LR: 0.01, T: 0.07000), Drift: 0.0895, Variance: 0.7893
Epoch 3: Loss 5.3498, Train Acc 0.7216, Val Acc 0.5167, Avg Grad Norm 2.20e-01, Train Time 0.67, Valid Time 0.05 (LR: 0.01, T: 0.07000), Drift: 0.1186, Variance: 0.7938
Epoch 4: Loss 5.2595, Train Acc 0.7485, Val Acc 0.5215, Avg Grad Norm 2.58e-01, Train Time 0.68, Valid Time 0.05 (LR: 0.01, T: 0.07000), Drift: 0.1432, Variance: 0.7985
Epoch 5: Loss 5.1854, Train Acc 0.7620, Val Acc 0.5184, Avg Grad Norm 2.90e-01, Train Time 0.68, Valid Time 0.07 (LR: 0.01, T: 0.07000), Drift: 0.1632, Variance: 0.8030
Epoch 6: Loss 5.1206, Train Acc 0.7743, Val Acc 0.5281, Avg Grad Norm 3.18e-01, Train Time 0.81, Valid Time 0.06 (LR: 0.01, T: 0.07000), Drift: 0.1806, Var

TypeError: expected Tensor as element 0 in argument 0, but got tuple